# 因子框架基本用法

从内存数据、数据库或其他因子取得基础数据，通过表达式和算子组合成衍生因子，再用 `readData()` 返回日期 × 证券的 `DataFrame`。最小例子如下；完整流程见 [快速入门](QuickStart.ipynb)。

同一张表的多个因子以 `Panel` 组织，轴顺序为“因子 × 时点 × 证券”。日期使用 `datetime`，证券 ID 和因子名使用字符串。

In [ ]:
import datetime as dt
import pandas as pd
from QuantStudio.Factor.Factor import DataFactor
from QuantStudio.Factor.BasicOperator import rename

ids = ["A", "B"]
dates = [dt.datetime(2025, 1, 1), dt.datetime(2025, 1, 2)]
price = DataFactor(pd.DataFrame([[10., 20.], [12., 22.]], index=dates, columns=ids))
scaled_price = rename(price / 10, factor_name="标准化价格")
print(scaled_price.readData(ids=ids, dts=dates))

```mermaid
graph TB
    subgraph FactorFramework["因子框架"]
        direction TB

        FactorDB["因子库<br/>FactorDB"]

        FactorTable1["因子表1<br/>FactorTable"]
        FactorTable2["因子表2<br/>FactorTable"]

        subgraph FT1["因子表1内容"]
            F11["因子1<br/>Factor"]
            F12["因子2<br/>Factor"]
            F1M["因子M<br/>Factor"]
        end

        subgraph FT2["因子表2内容"]
            F21["因子1<br/>Factor"]
            F22["因子2<br/>Factor"]
            F2N["因子N<br/>Factor"]
        end
        
    end
    
    FactorDB --> FactorTable1
    FactorDB --> FactorTable2
    
    FactorTable1 --> F11
    FactorTable1 --> F12
    FactorTable1 --> F1M
    
    FactorTable2 --> F21
    FactorTable2 --> F22
    FactorTable2 --> F2N
    
    DF["DataFrame<br/>- index: 时间<br/>- columns: 证券代码"]
    
    F11 --> DF
    F12 --> DF
    F1M --> DF
    F21 --> DF
    F22 --> DF
    F2N --> DF

    style FactorFramework fill:#e1f5fe
    style FactorDB fill:#f87f89
    style FactorTable1 fill:#efbaf7
    style FactorTable2 fill:#efbaf7
    style DF fill:#fff3e0
```

# 因子与计算图

因子框架构建在计算图框架之上。每个因子（`Factor`）继承自计算图节点（`Node`），因此单个因子本质上是一个计算节点；多个因子之间的依赖关系构成有向无环计算图（DAG）。因子的数据获取最终由计算引擎驱动执行。

关于计算图的节点生命周期、Context、引擎调度等核心概念，请参见 **[计算图框架](../Core/计算图框架.ipynb)**；关于不同计算引擎的选择和并行策略，请参见 **[计算引擎](../Core/计算引擎.ipynb)**。

## 因子的两类角色

因子在计算图中扮演两种角色：

* **基础因子**：不依赖其他因子，数据直接来源于原始数据的因子。例如，通过 `FactorTable.getFactor()` 从因子表中获取的因子，以及通过 `DataFactor` 直接给入字面量数据的因子。基础因子是计算图的叶子节点。
* **衍生因子**：依赖其他因子（称为**描述子**），通过因子运算得到的因子。例如，市净率（PB）因子由总市值因子除以股东权益因子运算得到。衍生因子是计算图的内部节点，由算子（`FactorOperator`）作用于描述子产生。

## 因子上下文对象

因子框架在通用计算图上下文的基础上，定义了两个专用的上下文类型：

| 类型 | 基类 | 用途 |
|------|------|------|
| `FactorContext` | `Context` | 因子计算的全局上下文，新增 `DTRuler`（时点标尺）、`SectionIDs`（默认截面 ID）和 `DataCache`（因子数据缓存）字段 |
| `FactorLocalContext` | `DTLocalContext` | 因子计算的局部上下文，携带当前计算时点、 ID 序列和截面 ID信息 |

这两个类型定义在 `QuantStudio.Factor.Factor` 模块中，但在因子框架的各个组件（FactorTable、Factor、FactorOperator）中均被广泛使用。

# 因子运算

## 四类运算概述

因子的运算按照数据依赖范围划分为四类，每种运算的适用范围和效率各不相同：

| 运算类型 | 算子类 | 衍生因子类 | 依赖范围 | 典型应用 |
|----------|--------|------------|----------|----------|
| **单点运算** | `PointOperator` | `PointOperation` | 同时间点、同证券 | 估值指标（PB、PE 等） |
| **时序运算** | `TimeOperator` | `TimeOperation` | 历史时间序列、同证券 | 移动平均、EMA |
| **截面运算** | `SectionOperator` | `SectionOperation` | 同时间点、全截面证券 | 数据标准化（Z-score） |
| **面板运算** | `PanelOperator` | `PanelOperation` | 历史时间序列 + 全截面证券 | 时序截面双重标准化 |

每种因子的运算本质上是一个可调用对象（算子 `FactorOperator`），算子作用于描述子（依赖因子）后产生新的衍生因子。这些运算可以嵌套组合以形成更复杂的计算图。

详细的因子运算使用说明、参数配置和代码示例，请参见 **[因子开发](因子开发.ipynb)**。

## 运算符重载

`Factor` 类重载了常用 Python 运算符（`+`、`-`、`*`、`/`、`<`、`>`、`==`、`&`、`|`、`~` 等），使其可以直接作用于因子对象产生衍生因子。这些重载运算符定义在 `QuantStudio.Factor.BasicOperator` 中，本质上是基于单点运算的快捷方式。

`BasicOperator` 还提供了 `rename` 算子用于给因子重命名。

## 内置算子

`QuantStudio.Factor.FactorOperator` 模块预定义了一些常用的内置算子（如 `Log`、`RollingMean`、`SectionRank` 等），可直接使用。

# 常用接口

| 目标 | 调用 | 返回 |
|---|---|---|
| 发现表 | `fdb.TableNames` | 表名列表（属性） |
| 获取表 | `fdb.getTable(table_name, args={...})` | `FactorTable` |
| 发现字段 | `table.FactorNames` | 因子名列表（属性） |
| 获取因子 | `table.getFactor(factor_name, args={...})` | `Factor` |
| 读取多因子 | `table.readData(factor_names=..., ids=..., dts=...)` | `Panel` |
| 读取单因子 | `factor.readData(ids=..., dts=...)` | `DataFrame` |
| 查询范围 | `getID()`、`getDateTime()` | ID 或时点列表 |
| 查询说明 | `getMetaData()`、`getFactorMetaData()` | 元信息 |

`DataFactor` 可直接接收标量、Series 或 DataFrame。日期、证券超出已有数据的部分按其对齐规则返回缺失值，标量可广播；不应将其理解为无条件返回原数据。

# 可用因子库

QuantStudio 提供了多种因子库实现，用于连接不同的数据源：

| 因子库 | 模块 | 数据源 | 可写 | 文档 |
|--------|------|--------|------|------|
| `JYDB` | `QuantStudio.Factor.JYDB` | 聚源关系数据库 | 否 | [JYDB](JYDB.ipynb) |
| `SQLDB` | `QuantStudio.Factor.SQLDB` | 通用 SQL 数据库 | 是 | [SQLDB](SQLDB.ipynb) |
| `HDF5DB` | `QuantStudio.Factor.HDF5DB` | 本地 HDF5 文件 | 是 | [HDF5DB](HDF5DB.ipynb) |
| `BaoStockDB` | `QuantStudio.Factor.BaoStockDB` | BaoStock 在线 API | 否 | [BaoStockDB](BaoStockDB.ipynb) |

各因子库的具体连接方式、配置文件格式、支持的表类型和参数说明，请参见对应的文档。

# 计算结果持久化

`FactorStorer` 将依赖因子的计算结果写入可写因子库。参数包括 `TargetFDB`、`TargetTable` 和 `IfExists`。将存储节点作为引擎目标，依赖的因子由引擎自动计算。

只需要一次保存时，可把计算得到的 DataFrame 组合成 `Panel`，调用目标库 `writeData()`。读写模式与实际示例见 [HDF5DB](HDF5DB.ipynb)，后台运行任务可接入 [缓存](../Core/缓存.ipynb)。

# 相关文档索引

| 文档 | 内容 |
|------|------|
| [QuickStart](QuickStart.ipynb) | 因子框架快速入门示例 |
| [因子开发](因子开发.ipynb) | 因子运算详解（四类运算、算子定义、参数说明） |
| [JYDB](JYDB.ipynb) | 聚源数据库因子库的使用 |
| [HDF5DB](HDF5DB.ipynb) | 本地 HDF5 因子库的使用 |
| [SQLDB](SQLDB.ipynb) | 通用 SQL 因子库的使用 |
| [BaoStockDB](BaoStockDB.ipynb) | BaoStock 因子库的使用 |
| [计算图框架](../Core/计算图框架.ipynb) | 计算图节点、Context、生命周期 |
| [计算引擎](../Core/计算引擎.ipynb) | 引擎类型、并行策略 |
| [缓存](../Core/缓存.ipynb) | 因子缓存机制 |

存储布局、SQL 元信息和算子绑定机制见 [原理和细节](原理和细节.ipynb)。